In [2]:
import duckdb
import ollama
import re

# Setup
DB_PATH = "../data/omop_clinical.duckdb"
MODEL_NAME = "qwen2.5-coder:7b"

def get_unmapped_drugs(limit=5):
    """Fetch a sample of unique unmapped medication records from DuckDB."""
    print("🔌 Connecting to DuckDB to fetch unmapped medications...")
    with duckdb.connect(DB_PATH) as con:
        query = """
            SELECT DISTINCT drug_source_value
            FROM drug_exposure
            WHERE drug_concept_id = 0
            AND drug_source_value IS NOT NULL
            AND drug_source_value != 'Unknown' -- 🚨 O FILTRO PARA IGNORAR LIXO
            LIMIT ?
        """
        return [row[0] for row in con.execute(query, [limit]).fetchall()]

def ai_drug_normalization(raw_term):
    """Uses local LLM to extract the clean, generic active ingredient from messy medication text."""
    system_prompt = """
    You are an expert Clinical Pharmacist and Data Informatician.
    Your task is to normalize raw medication text into a clean, generic active ingredient name.
    RULES:
    1. Respond ONLY with the generic medication name (active ingredient).
    2. Strip away any brand names, dosages (e.g., '500 MG'), forms (e.g., 'Tablet', 'Capsule'), and numeric IDs.
    3. Keep it as short and precise as possible.
    """
    
    try:
        response = ollama.chat(
            model=MODEL_NAME,
            messages=[
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': f"Raw medication text: '{raw_term}'"}
            ]
        )
        
        # Clean the output: remove quotes and extra whitespace
        clean_text = response['message']['content'].strip().strip("'").strip('"')
        return clean_text
    except Exception as e:
        return f"Error: {e}"

# EXECUTION BLOCK
print("⚙️ STARTING AI-ASSISTED DRUG MAPPING (RXNORM FUZZY SEARCH)\n" + "-"*50)

unmapped_drugs = get_unmapped_drugs(limit=5)

if not unmapped_drugs:
    print("✅ No unmapped medications found! Everything is perfect.")
else:
    print(f"⚠️ Found unmapped medications. Sending a sample of {len(unmapped_drugs)} to Qwen...\n")
    
    for term in unmapped_drugs:
        print(f"🔸 Raw Drug (Failed): {term}")
        
        # 1. Ask AI to extract the active ingredient
        normalized_drug = ai_drug_normalization(term)
        print(f"✨ AI Suggestion:     {normalized_drug}")
        
        # 2. Search DuckDB using Fuzzy Matching in the RxNorm vocabulary
        with duckdb.connect(DB_PATH) as con:
            search_query = """
                SELECT concept_id, concept_name, concept_class_id 
                FROM concept 
                WHERE vocabulary_id = 'RxNorm' 
                AND LOWER(concept_name) LIKE LOWER(?)
                LIMIT 1
            """
            # Using % wildcards to find the active ingredient anywhere in the concept name
            match = con.execute(search_query, [f"%{normalized_drug}%"]).fetchone()
            
            if match:
                print(f"🎯 DB Match Found!   ID: {match[0]} | Name: '{match[1]}' | Class: {match[2]}")
            else:
                print("❌ Still no match. Might be a complex multi-ingredient drug or missing in standard vocab.")
        print("-" * 50)

⚙️ STARTING AI-ASSISTED DRUG MAPPING (RXNORM FUZZY SEARCH)
--------------------------------------------------
🔌 Connecting to DuckDB to fetch unmapped medications...
✅ No unmapped medications found! Everything is perfect.
